# Demo notebook for Model Reader

In [1]:
model = 'NRLMSIS'
variables_requested = ['T_n', 'rho']

In [2]:
from pathlib import Path

# Path.cwd() gets the 'Validation/Notebooks' directory and .parent.parent moves up two levels to the repository root.
root_dir = Path.cwd().parent.parent

# Make sure test data has been downloaded.
download = str(root_dir / 'download_TestData.py')
%run $download $model

# Construct the absolute path to the data and add the trailing slash (or direct to your own data)
file_dir = str(root_dir / 'tests' / 'TestData' / model) + '/'

Target data directory: /Users/ddezeeuw/gitDBG/July2026/Kamodo/tests/TestData

[✓] NRLMSIS data already exists in /Users/ddezeeuw/gitDBG/July2026/Kamodo/tests/TestData/NRLMSIS. Skipping.


In [3]:
import kamodo_ccmc.flythrough.model_wrapper as MW

In [4]:
# Check for all variables containing 'temperature' in the base model dictionary
MW.Variable_Search('temperature', model)

T_n: ['neutral temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'height'], 'K']
T_exo: ['exospheric temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'height'], 'K']


In [5]:
# Check for all variables containing 'temperature' in these specific model output files
MW.Variable_Search('temperature', model, file_dir)

T_n: ['neutral temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'height'], 'K']
T_exo: ['exospheric temperature', 'GDZ-sph', ['time', 'lon', 'lat', 'height'], 'K']


In [6]:
# Check selected variable units
MW.Var_units(model, variables_requested)

{'rho': 'g/cm**3', 'T_n': 'K'}

In [7]:
# Confirm time method works with model reader
MW.File_Times(model, file_dir)

UTC time ranges
------------------------------------------
Start Date: 2025-09-17  Time: 00:00:00
End Date: 2025-09-18  Time: 23:45:00


(datetime.datetime(2025, 9, 17, 0, 0, tzinfo=datetime.timezone.utc),
 datetime.datetime(2025, 9, 18, 23, 45, tzinfo=datetime.timezone.utc))

In [8]:
# Confirm file list method works with model reader
MW.File_List(model, file_dir)

'/Users/ddezeeuw/gitDBG/July2026/Kamodo/tests/TestData/NRLMSIS/NRLMSIS00.3D.2025260.nc,/Users/ddezeeuw/gitDBG/July2026/Kamodo/tests/TestData/NRLMSIS/NRLMSIS00.3D.2025261.nc'

In [9]:
# Check that time files creation works, that reader works for one variable,
# and that an unknown variable request does not break it.
from os.path import isfile
from os import remove

if isfile(file_dir+model+'_times.txt'):
    remove(file_dir+model+'_times.txt')
if isfile(file_dir+model+'_list.txt'):
    remove(file_dir+model+'_list.txt')

reader = MW.Model_Reader(model)
kamodo_object = reader(file_dir, variables_requested=['Trash'])
kamodo_object

Creating the time files...done.
Variable name(s) not recognized: ['Trash']


{}

In [10]:
# Check that reader works for one variable with an unknown variable
kamodo_object = reader(file_dir, variables_requested=['Trash', variables_requested[0]])
kamodo_object

Variable name(s) not recognized: ['Trash']


{T_n(rvec_GDZsph4D): <function multitime_interp.<locals>.interp at 0x133543b60>, T_n: <function multitime_interp.<locals>.interp at 0x133543b60>, T_n_ijk(time, lon, lat, height): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x133638040>, T_n_ijk: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x133638040>}

In [11]:
# Check that reading the time files works and that reader works for one variable,
kamodo_object = reader(file_dir, variables_requested=variables_requested[:1])
kamodo_object

{T_n(rvec_GDZsph4D): <function multitime_interp.<locals>.interp at 0x133638670>, T_n: <function multitime_interp.<locals>.interp at 0x133638670>, T_n_ijk(time, lon, lat, height): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x133638d50>, T_n_ijk: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x133638d50>}

In [12]:
# Confirm that interpolation works. 
from math import isnan
print(kamodo_object.T_n([5.2, 10., 60., 350.]))
if isnan(kamodo_object.T_n([5.2, 10., 60., 350.])[0]):
    raise AttributeError('Returned value is a NaN.')
else:
    print('Value is valid.')
print(kamodo_object.T_n_ijk(time=5.2, lon=10., lat=60., height=350.))
if isnan(kamodo_object.T_n_ijk(time=5.2, lon=10., lat=60., height=350.)):
    raise AttributeError('Returned value is a NaN.')
else:
    print('Value is valid.')
if not kamodo_object.T_n([5.2, 10., 60., 350.]) == kamodo_object.T_n_ijk(time=5.2, lon=10., lat=60., height=350.):
    raise AttributeError('Values are not equal.')
else:
    print('Values are equal.')
print(kamodo_object.T_n_ijk(time=5.2, lon=10).shape)

[1062.26568604]
Value is valid.
1062.2656860351562
Value is valid.
Values are equal.
(37, 101)


In [30]:
# Check that reading the time files works, and that the reader works for all variables
kamodo_object = reader(file_dir)
kamodo_object

{N_O(rvec_GDZsph4D): <function multitime_interp.<locals>.interp at 0x1373b80f0>, N_O: <function multitime_interp.<locals>.interp at 0x1373b80f0>, N_O_ijk(time, lon, lat, height): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x1373b8670>, N_O_ijk: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x1373b8670>, N_N2(rvec_GDZsph4D): <function multitime_interp.<locals>.interp at 0x1373b9170>, N_N2: <function multitime_interp.<locals>.interp at 0x1373b9170>, N_N2_ijk(time, lon, lat, height): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x1373b94e0>, N_N2_ijk: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x1373b94e0>, N_O2(rvec_GDZsph4D): <function multitime_interp.<locals>.interp at 0x1373ba350>, N_O2: <function multitime_interp.<locals>.interp at 0x1373ba350>, N_O2_ijk(time, lon, lat, height): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x1373baa30>, N_O2_ijk: <function gridify.<locals>.decorato

In [14]:
# Get a list of all of the functionalized variables, both regular and gridded
var_list = list(MW.Variable_Search('', model, file_dir, return_dict=True).keys())
varijk_list = sorted(var_list + [item+'_ijk' for item in var_list])
varijk_list

['N_AO',
 'N_AO_ijk',
 'N_Ar',
 'N_Ar_ijk',
 'N_H',
 'N_H_ijk',
 'N_He',
 'N_He_ijk',
 'N_N',
 'N_N2',
 'N_N2_ijk',
 'N_N_ijk',
 'N_O',
 'N_O2',
 'N_O2_ijk',
 'N_O_ijk',
 'T_exo',
 'T_exo_ijk',
 'T_n',
 'T_n_ijk',
 'rho',
 'rho_ijk']

In [15]:
# Test coordinate range logic for all variables
MW.Coord_Range(kamodo_object, varijk_list)

The minimum and maximum values for each variable and coordinate are:
N_AO:
time: [np.float64(0.0), np.float64(47.75), 'hr']
lon: [np.float64(-180.0), np.float64(180.0), 'deg']
lat: [np.float64(-90.0), np.float64(90.0), 'deg']
height: [np.float64(0.0), np.float64(500.0), 'km']

N_AO_ijk:
time: [np.float64(0.0), np.float64(47.75), 'hr']
lon: [np.float32(-180.0), np.float32(180.0), 'deg']
lat: [np.float32(-90.0), np.float32(90.0), 'deg']
height: [np.float32(0.0), np.float32(500.0), 'km']

N_Ar:
time: [np.float64(0.0), np.float64(47.75), 'hr']
lon: [np.float64(-180.0), np.float64(180.0), 'deg']
lat: [np.float64(-90.0), np.float64(90.0), 'deg']
height: [np.float64(0.0), np.float64(500.0), 'km']

N_Ar_ijk:
time: [np.float64(0.0), np.float64(47.75), 'hr']
lon: [np.float32(-180.0), np.float32(180.0), 'deg']
lat: [np.float32(-90.0), np.float32(90.0), 'deg']
height: [np.float32(0.0), np.float32(500.0), 'km']

N_H:
time: [np.float64(0.0), np.float64(47.75), 'hr']
lon: [np.float64(-180.0), np.floa

In [16]:
# Check that the kamodo object was built properly.
print(kamodo_object.T_n([5.2, 10., 60., 350.]))
if isnan(kamodo_object.T_n([5.2, 10., 60., 350.])[0]):
    raise AttributeError('Returned value is a NaN.')
else:
    print('Value is valid.')

[1062.26568604]
Value is valid.


In [17]:
# Check that the reader works for the testing subset
kamodo_object = reader(file_dir, variables_requested=variables_requested)
kamodo_object

{rho(rvec_GDZsph4D): <function multitime_interp.<locals>.interp at 0x13365aa30>, rho: <function multitime_interp.<locals>.interp at 0x13365aa30>, rho_ijk(time, lon, lat, height): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x133639640>, rho_ijk: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x133639640>, T_n(rvec_GDZsph4D): <function multitime_interp.<locals>.interp at 0x13365a140>, T_n: <function multitime_interp.<locals>.interp at 0x13365a140>, T_n_ijk(time, lon, lat, height): <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x13365a2a0>, T_n_ijk: <function gridify.<locals>.decorator_gridify.<locals>.wrapped at 0x13365a2a0>}

In [18]:
# Confirm that the interpolator works for each testing variable and type
# rho
print(kamodo_object.rho([5.2, 10., 60., 350.]))
print(kamodo_object.rho_ijk(time=5.2, lon=10., lat=60., height=350.))
if not kamodo_object.rho([5.2, 10., 60., 350.]) == kamodo_object.rho_ijk(time=5.2, lon=10., lat=60., height=350.):
    raise AttributeError('Values are not equal.')
else:
    print('Values are equal.')
print('Shape lat x alt: ',kamodo_object.rho_ijk(time=5.2, lon=10.).shape)
# T_n
print(kamodo_object.T_n([5.2, 10., 60., 350.]))
print(kamodo_object.T_n_ijk(time=5.2, lon=10., lat=60., height=350.))
if not kamodo_object.T_n([5.2, 10., 60., 350.]) == kamodo_object.T_n_ijk(time=5.2, lon=10., lat=60., height=350.):
    raise AttributeError('Values are not equal.')
else:
    print('Values are equal.')
print('Shape lat x alt: ',kamodo_object.T_n_ijk(time=5.2, lon=10.).shape)

[7.37126334e-15]
7.371263343457343e-15
Values are equal.
Shape lat x alt:  (37, 101)
[1062.26568604]
1062.2656860351562
Values are equal.
Shape lat x alt:  (37, 101)


In [19]:
# Make sure rendering works in jupyter notebooks
import plotly.io as pio
pio.renderers.default = 'iframe'

In [20]:
# Set time strings to label plots
from datetime import timedelta
import kamodo_ccmc.tools.timefunctions as tf
import kamodo_ccmc.tools.plotfunctions as pf
h = 40.
timetext = tf.timeDTtoSTR(kamodo_object.filedate + timedelta(hours=h))
tt = MW.File_Times(model, file_dir, print_output=False)  # get start and end datetimes
timetext2 = tf.timeDTtoSTR(tt[0])+' - '+tf.timeDTtoSTR(tt[1])
timetext, timetext2

('2025/09/18 16:00:00', '2025/09/17 00:00:00 - 2025/09/18 23:45:00')

In [21]:
# Generate a plot for validation
pf.figMods(kamodo_object.plot('rho_ijk', plot_partial={'rho_ijk': {'time': h, 'height': 250.}}),
           ncont=201, colorscale='Viridis', llText=timetext, llText2='time='+str(h)+', height=250.')

In [22]:
# Generate a plot for validation
pf.figMods(kamodo_object.plot('T_n_ijk', plot_partial={'T_n_ijk': {'time': h, 'height': 250.}}),
           ncont=201, colorscale='Viridis', llText=timetext, llText2='time='+str(h)+', height=250.')

In [23]:
# Generate a plot for validation
pf.figMods(kamodo_object.plot('T_n_ijk', plot_partial={'T_n_ijk': {'lat': 10., 'height': 200.}}),
           ncont=201, colorscale='Viridis', llText=timetext2, llText2='lat=10., height=200.')

In [37]:
# Generate a plot for validation
pf.figMods(kamodo_object.plot('T_exo_ijk', plot_partial={'T_exo_ijk': {'time': h, 'lon': 45., 'height': 250.}}),
           ncont=201, colorscale='Viridis', llText=timetext, llText2='time='+str(h)+', lon=45., height=250.')

In [24]:
# Test that more than one variable works through the flythrough
from kamodo_ccmc.flythrough import SatelliteFlythrough as SF
import datetime as dt
start_utcts = dt.datetime(2025, 9, 17, 0, 0).replace(tzinfo=dt.timezone.utc).timestamp()
end_utcts = dt.datetime(2025, 9, 18, 23, 45).replace(tzinfo=dt.timezone.utc).timestamp()-1
results = SF.ModelFlythrough(model, file_dir, [variables_requested[0]], [start_utcts, end_utcts], [0., 180.], [60., -60.],
                             [400., 400.], 'GDZ-sph')
results[variables_requested[0]]

{'T_n': 'K', 'utc_time': 's', 'net_idx': '', 'c1': 'deg', 'c2': 'deg', 'c3': 'km'}


array([1039.66320801, 1074.23474121])

In [25]:
# Test that one variable works through the flythrough
results = SF.ModelFlythrough(model, file_dir, [variables_requested[0]], [start_utcts], [0.], [60.],
                             [400.], 'GDZ-sph')
results[variables_requested[0]]

{'T_n': 'K', 'utc_time': 's', 'net_idx': '', 'c1': 'deg', 'c2': 'deg', 'c3': 'km'}


array([1039.66320801])